# EDA — Store Sales Time Series Forecasting

Quick exploratory analysis backing the modeling choices in `ARCHITECTURE.md`
(gradient boosting, calendar + lag features, oil price, holiday flags) and
useful for the presentation's technical deep dive.

Run `python scripts/setup_data.py` first so `data/raw/` is populated.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from demand_forecast.data.ingest import load_raw_tables, merge_dataset

RAW_DIR = Path.cwd().parent / "data" / "raw"
tables = load_raw_tables(RAW_DIR)
train = tables["train"]
train.head()

## Total sales over time — trend, seasonality, and the 2016 earthquake dip

In [ ]:
daily_total = train.groupby("date")["sales"].sum()
daily_total.plot(figsize=(12, 4), title="Total daily sales across all stores/families")
plt.ylabel("Total units sold")
plt.show()

## Which product families drive the most volume?

In [ ]:
top_families = train.groupby("family")["sales"].sum().sort_values(ascending=False).head(10)
top_families.plot(kind="barh", figsize=(8, 5), title="Top 10 families by total sales")
plt.gca().invert_yaxis()
plt.xlabel("Total units sold (2013-2017)")
plt.show()

## Day-of-week seasonality (motivates the lag-7 / rolling-7 features)

In [ ]:
by_dow = train.assign(dow=train["date"].dt.day_name()).groupby("dow")["sales"].mean()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_dow.reindex(order).plot(kind="bar", figsize=(8, 4), title="Average sales by day of week")
plt.ylabel("Mean units sold per (store, family, day)")
plt.show()

## Oil price vs. total sales (Ecuador is oil-export dependent — motivates including `oil_price` as a feature)

In [ ]:
oil = tables["oil"].rename(columns={"dcoilwtico": "oil_price"}).set_index("date")["oil_price"].ffill()
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(daily_total.index, daily_total.values, color="tab:blue", label="Total sales")
ax1.set_ylabel("Total sales", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(oil.index, oil.values, color="tab:orange", label="Oil price (WTI)")
ax2.set_ylabel("Oil price (USD)", color="tab:orange")
plt.title("Total sales vs. oil price")
plt.show()

## Store type / cluster spread (motivates the fairness segment analysis in `docs/RESPONSIBLE_AI.md`)

In [ ]:
merged = merge_dataset(tables, split="train")
by_type = merged.groupby("type")["sales"].agg(["mean", "std", "count"])
by_type